# CS313 Nhóm 5 – Giai đoạn 1: cài môi trường và chạy thử DAMR trên PSM 1 epoch

**Trước khi chạy:**
1. *Settings → Accelerator*: chọn **GPU T4** (PyTorch mới trên Kaggle không còn hỗ trợ P100).
2. *Settings → Internet*: **On** (để clone repo và cài thư viện).
3. *Add Input*: thêm bộ dữ liệu chứa PSM (và MSL, SMD nếu có). Notebook tự tìm file theo tên, không cần đúng tên thư mục:
   `train.csv`, `test.csv`, `test_label.csv` (PSM) · `MSL_train.npy`, `MSL_test.npy`, `MSL_test_label.npy` · `SMD_train.npy`, `SMD_test.npy`, `SMD_test_label.npy`.
   Bản chuẩn là thư mục `all_datasets` của Time-Series-Library (TSLib); nếu nhóm chưa có trên Kaggle thì tải từ Google Drive của TSLib rồi tạo một Kaggle Dataset riêng.

In [ ]:
REPO = "https://github.com/Waterbear566/cs313-damr-reproduce.git"
BRANCH = "pipeline-common"   # đổi thành "main" sau khi merge PR #3
!rm -rf /kaggle/working/repo && git clone -q -b $BRANCH $REPO /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1

## 1. Môi trường (dòng 5 trong checklist)

In [ ]:
# Kaggle đã có torch, numpy, pandas, scikit-learn. Chỉ cài phần còn thiếu, KHÔNG cài lại torch.
!pip install -q thop faiss-cpu lightning timm 2>&1 | tail -2
import sys, platform, subprocess, torch, numpy, sklearn, pandas
print("Python   ", platform.python_version())
print("PyTorch  ", torch.__version__, "| CUDA", torch.version.cuda, "| cuDNN", torch.backends.cudnn.version())
print("numpy    ", numpy.__version__, "| pandas", pandas.__version__, "| sklearn", sklearn.__version__)
print("GPU      ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
print(subprocess.run(["nvidia-smi","--query-gpu=name,driver_version,memory.total","--format=csv"],capture_output=True,text=True).stdout)
!pip freeze > /kaggle/working/environment_kaggle.txt

## 2. Dữ liệu (dòng 6): tìm file trong /kaggle/input và nối vào `DAMR/data/`

In [ ]:
import glob, os
NEED = {"psm": ["train.csv", "test.csv", "test_label.csv"],
        "msl": ["MSL_train.npy", "MSL_test.npy", "MSL_test_label.npy"],
        "smd": ["SMD_train.npy", "SMD_test.npy", "SMD_test_label.npy"]}
for ds, files in NEED.items():
    # thư mục nào chứa đủ cả 3 file thì dùng thư mục đó
    cands = [d for d in {os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/{files[0]}", recursive=True)}
             if all(os.path.exists(os.path.join(d, f)) for f in files)]
    if ds == "psm":   # tên train.csv rất phổ biến: ưu tiên thư mục có chữ PSM
        cands = sorted(cands, key=lambda d: "psm" not in d.lower())
    if not cands:
        print(f"{ds.upper()}: chưa thấy trong /kaggle/input"); continue
    os.makedirs(f"DAMR/data/{ds}", exist_ok=True)
    for f in files:
        dst = f"DAMR/data/{ds}/{f}"
        if not os.path.exists(dst): os.symlink(os.path.join(cands[0], f), dst)
    print(f"{ds.upper()}: dùng {cands[0]}" + (f"  (còn {len(cands)-1} chỗ khác)" if len(cands) > 1 else ""))
!python scripts/check_data.py --root DAMR/data --datasets PSM

## 3. Chạy thử DAMR trên PSM 1 epoch (dòng 7)
Dùng đúng siêu tham số của tác giả (`lr=1e-4`, batch 512, cửa sổ 60 bước 10), chỉ giảm số epoch xuống 1 và bật ghi log mở rộng.

In [ ]:
import time
t0 = time.time()
!cd DAMR && python main.py --dataset=PSM --data_path=./data/psm --seed=0 --device=cuda:0 --lr=1e-4 --epochs=1 \
    --log_extra=True --save_dir=../checkpoints/smoke --results_csv=../results/smoke_PSM.csv 2>&1 | tee ../logs/smoke_PSM_1epoch.log
wall = time.time() - t0
print(f"Tổng thời gian (kể cả nạp dữ liệu): {wall:.0f} s")

In [ ]:
import pandas as pd, re
print(pd.read_csv("results/smoke_PSM.csv"))
log = open("logs/smoke_PSM_1epoch.log", encoding="utf-8").read()
epoch_s = float(re.findall(r"Epoch: 1/1, Time: ([\d.]+)", log)[0])
print(f"\n1 epoch (train + chấm test) = {epoch_s:.0f} s")
print(f"Ước lượng E2 trên PSM: 40 epoch × 3 seed ≈ {epoch_s*40*3/3600:.1f} giờ GPU")
print(f"Ước lượng E4 trên PSM: 40 epoch × 5 biến thể × 3 seed ≈ {epoch_s*40*15/3600:.1f} giờ GPU")
print("(MSL, SMD có số chiều và độ dài khác nên cần đo riêng.)")

## 4. Lưu kết quả
Tải về `logs/smoke_PSM_1epoch.log`, `results/smoke_PSM.csv` và `environment_kaggle.txt`, rồi điền phiên bản Python, PyTorch, CUDA và GPU vào README. Chạy thật E2 bằng: `!bash scripts/run_e2.sh PSM 0`.

In [ ]:
!cp -r logs results /kaggle/working/ 2>/dev/null; ls -la /kaggle/working